# ARC-AGI-3 - Slim Submission Notebook (Attached Skills Bundle)

This notebook loads the modular solver skills from the attached Kaggle dataset (`arc3x-skills-bundle`).
It requires **zero massive code bloat** in the notebook itself.

### Attached Datasets Required:
1. `competitions/arc-prize-2026-arc-agi-3` (Official competition environment files & wheels)
2. `samrishb/arc3x-skills-bundle` (or uploaded zip containing `arc3x/` and `world_model_lab/`)


In [ ]:
# ---------------------------------------------------------------------------
# Cell 1: Environment Setup, Dataset Linking & Baseline Submission Creation
# ---------------------------------------------------------------------------
import os, sys, glob, json, time, subprocess
from pathlib import Path
import pandas as pd

os.environ.setdefault("ONLY_RESET_LEVELS", "true")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# 1. Immediately create baseline submission files so Kaggle validator passes at any point
init_sub = pd.DataFrame([["1_0", "1", True, 1.0]], columns=["row_id", "game_id", "end_of_game", "score"])
init_sub.to_parquet(WORKING_DIR / "submission.parquet", index=False)
init_sub.to_csv(WORKING_DIR / "submission.csv", index=False)
print("SUCCESS: Initialized baseline submission.parquet and submission.csv")

# 2. Find and mount the arc3x skills bundle
def find_input(*names):
    for base in ("/kaggle/input", "."):
        for n in names:
            hits = glob.glob(f"{base}/**/{n}", recursive=True)
            if hits:
                return sorted(hits, key=len)[0]
    return None

ENV_DIR = find_input("environment_files")
print("environment_files:", ENV_DIR)

# Locate attached arc3x skills
arc3x_hit = find_input("arc3x")
if arc3x_hit:
    parent_dir = str(Path(arc3x_hit).parent)
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)
    print(f"Loaded skills bundle from: {parent_dir}")
else:
    sys.path.insert(0, "/kaggle/working")

# 3. Install wheels if needed
try:
    import arc_agi  # noqa: F401
    print("arc_agi already importable")
except ImportError:
    for pat in ("arc_agi*.whl", "arcengine*.whl", "re_arc*.whl"):
        for w in glob.glob(f"/kaggle/input/**/{pat}", recursive=True):
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", w], check=False)
    import arc_agi  # noqa: F401

print("Ready!")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 2: Phase 1 - Plan Verification & Multi-Agent Dialectical Search
# ---------------------------------------------------------------------------
import time
from arc3x.explore import discover_games, solve_game
from arc3x.twin import Twin, Act
from arc3x.debate import DialecticalDebateAgent

# Locate pre-computed plans from attached dataset or working
plans_file = find_input("plans.json")
existing_plans = {}
if plans_file:
    try:
        blob = json.loads(Path(plans_file).read_text(encoding="utf-8"))
        for r in blob.get("results", []):
            if r.get("plan"):
                existing_plans[r["game_id"]] = r
        print(f"Loaded {len(existing_plans)} pre-computed family plans from {plans_file}")
    except Exception as e:
        print(f"Error reading plans: {e}")

games = discover_games(Path(ENV_DIR)) if ENV_DIR else []
print(f"Discovered {len(games)} local game families.")

verified_results = []
t0 = time.perf_counter()

for gid in games:
    r = existing_plans.get(gid)
    if r and r.get("plan"):
        try:
            tw = Twin(gid, Path(ENV_DIR))
            obs = tw.replay([Act(a[0], a[1], a[2]) for a in r["plan"]])
            verified_results.append(r)
            print(f"  {gid:16s} [VERIFIED] solved {r['levels_solved']}/{r['n_levels']} score: {r['est_score']:6.2f}")
        except Exception as exc:
            print(f"  {gid:16s} verification failed ({exc}); will re-search")
            r = None

    if r is None:
        budget = float(os.environ.get("ARC3X_BUDGET", 10.0))
        try:
            sol = solve_game(gid, env_dir=Path(ENV_DIR), budget_s=budget, verbose=False)
            res_dict = {
                "game_id": sol.game_id,
                "plan": [[a.aid, a.x, a.y] for a in sol.plan],
                "actions_per_level": sol.actions_per_level,
                "baselines": sol.baselines,
                "levels_solved": sol.levels_solved,
                "n_levels": len(sol.baselines),
                "est_score": sol.est_score,
                "steps": sol.steps,
                "seconds": sol.seconds,
            }
            verified_results.append(res_dict)
            print(f"  {gid:16s} [SEARCHED] solved {sol.levels_solved}/{len(sol.baselines)} score: {sol.est_score:6.2f}")
        except Exception as exc:
            print(f"  {gid:16s} search error: {exc}")

mean_sc = sum(r.get("est_score", 0) for r in verified_results) / max(1, len(verified_results))
print(f"\nPhase 1 verified mean estimated score: {mean_sc:.3f} ({(time.perf_counter()-t0):.1f}s)")
json.dump({"mean_est_score": mean_sc, "results": verified_results}, open(WORKING_DIR / "plans.json", "w"))

# Update submission files with Phase 1 scores
p1_records = [{"row_id": f"{r['game_id']}_0", "game_id": r['game_id'], "end_of_game": True, "score": float(r['est_score'])} for r in verified_results]
if p1_records:
    pd.DataFrame(p1_records).to_parquet(WORKING_DIR / "submission.parquet", index=False)
    pd.DataFrame(p1_records).to_csv(WORKING_DIR / "submission.csv", index=False)


In [ ]:
# ---------------------------------------------------------------------------
# Cell 3: Phase 2 - Graded Gateway Replay & Final Submission Generation
# ---------------------------------------------------------------------------
import socket
from urllib.parse import urlparse
import numpy as np

from arc3x.runner import build_families, gateway_as_graded, twin_as_graded, play_game
from arc3x.debate import DialecticalDebateAgent

BASE_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001")
ACTION_CAP = int(os.environ.get("ARC3X_ACTION_CAP", 800))
IS_RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}

families = build_families(Path(ENV_DIR) if ENV_DIR else None, plans_path=WORKING_DIR / "plans.json")
print(f"{len(families)} families loaded, {sum(1 for f in families if f.plan)} with plans")

debate_agent = DialecticalDebateAgent(seed=42)
student = None
try:
    from arc3x.student import Student
    sp = find_input("student*.npz")
    if sp:
        student = Student.load(sp)
        print(f"student policy loaded from {sp}")
except Exception as exc:
    print(f"no student policy ({type(exc).__name__}); fallback is Dialectical Debate agent")

def check_gateway(url: str, timeout: float = 3.0) -> bool:
    try:
        p = urlparse(url)
        host = p.hostname or "gateway"
        port = p.port or 8001
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except Exception:
        return False

has_gateway = check_gateway(BASE_URL, timeout=2.0)
if not has_gateway and IS_RERUN:
    print("Waiting for competition gateway to become ready...")
    for _ in range(15):
        time.sleep(3)
        if check_gateway(BASE_URL, timeout=2.0):
            has_gateway = True
            break

played, rng = [], np.random.default_rng(0)

try:
    if has_gateway:
        print(f"Connected to live competition gateway at {BASE_URL}")
        import arc_agi
        from arc_agi import OperationMode

        arcade = arc_agi.Arcade(
            operation_mode=OperationMode.COMPETITION,
            arc_base_url=BASE_URL,
            environments_dir="",
        )
        card = arcade.create_scorecard()
        envs = arcade.get_environments()
        print(f"gateway offers {len(envs)} runs")

        for i, info in enumerate(envs):
            gid = info.game_id
            try:
                env = arcade.make(gid, scorecard_id=card)
                if env is None:
                    print(f"  [{i+1}/{len(envs)}] {gid}: make() returned None"); continue
                res = play_game(gateway_as_graded(env), families, graded_game_id=gid,
                                action_cap=ACTION_CAP, student=student, rng=rng)
                played.append(res)
                print(f"  [{i+1}/{len(envs)}] {gid} fam={res.family} via={res.how} "
                      f"src={res.source} actions={res.actions_used} levels={res.levels_reached}")
            except Exception as exc:
                print(f"  [{i+1}/{len(envs)}] {gid}: {type(exc).__name__}: {exc}")

        try:
            arcade.close_scorecard(card)
            print("Scorecard closed and submitted successfully.")
        except Exception as exc:
            print(f"close_scorecard: {type(exc).__name__}: {exc}")

    else:
        print("No competition gateway detected (interactive draft mode).")
        print("Running offline self-verification across all 25 game families...")
        test_games = discover_games(Path(ENV_DIR)) if ENV_DIR else []
        for i, gid in enumerate(test_games):
            try:
                graded = twin_as_graded(gid, Path(ENV_DIR))
                res = play_game(graded, families, graded_game_id=gid,
                                action_cap=ACTION_CAP, student=student, rng=rng)
                played.append(res)
                print(f"  [{i+1}/{len(test_games)}] {gid} fam={res.family} via={res.how} "
                      f"src={res.source} actions={res.actions_used} levels={res.levels_reached}")
            except Exception as exc:
                print(f"  [{i+1}/{len(test_games)}] {gid}: {type(exc).__name__}: {exc}")

    print(f"\nPhase 2 finished successfully: played {len(played)} runs.")

finally:
    # Flushes the final official submission file
    sub_records = []
    for r in played:
        sub_records.append({
            "row_id": f"{r.graded_game_id}_0",
            "game_id": str(r.graded_game_id),
            "end_of_game": True,
            "score": float(r.levels_reached)
        })

    if not sub_records and (WORKING_DIR / "plans.json").exists():
        try:
            b = json.loads((WORKING_DIR / "plans.json").read_text())
            for r in b.get("results", []):
                sub_records.append({
                    "row_id": f"{r['game_id']}_0",
                    "game_id": str(r['game_id']),
                    "end_of_game": True,
                    "score": float(r.get("levels_solved", 1.0))
                })
        except Exception:
            pass

    if not sub_records:
        sub_records.append({"row_id": "1_0", "game_id": "1", "end_of_game": True, "score": 1.0})

    sub_df = pd.DataFrame(sub_records)
    sub_df.to_parquet(WORKING_DIR / "submission.parquet", index=False)
    sub_df.to_csv(WORKING_DIR / "submission.csv", index=False)
    print(f"SUCCESS: Finalized {len(sub_df)} rows in /kaggle/working/submission.parquet and submission.csv!")
